# Train set

## imports

In [2]:
import pandas as pd

## merge diffusion on the train split

In [3]:

all_subjects = pd.read_csv("../data/all_subjects_tract_means.csv")
train_split = pd.read_csv("../data/train_split.csv")

# keep only subjects that are in train_split
filtered = all_subjects[all_subjects["subjectID"].isin(train_split["Subject"])]

# add the train columns
merged = filtered.merge(
    train_split,
    left_on="subjectID",
    right_on="Subject",
    how="left"
)


merged = merged.drop(columns=["Subject"])

## double check counts

In [4]:
all_subjects["subjectID"].nunique()

1041

In [5]:
train_split["Subject"].nunique()

691

In [6]:
merged["subjectID"].nunique()

691

In [9]:
dd_df = pd.read_csv("../data/DD_AUC_targets.csv")


In [10]:
dd_df["Subject"].nunique()

889

## Merge the uptaded train split on delay discounting dataframe

In [11]:


merged = merged.merge(
    dd_df[["Subject", "DDisc_AUC_40K"]],
    left_on="subjectID",
    right_on="Subject",
    how="left"
)

merged = merged.drop(columns=["Subject"])

## Re-order merged df

In [12]:
cols = ["subjectID", "age_group", "strata", "Gender", "DDisc_AUC_40K"]
merged = merged[cols + [c for c in merged.columns if c not in cols]]

In [13]:
diffusion_subjects_df = (
all_subjects[["subjectID"]]
.drop_duplicates()
.sort_values("subjectID")
.copy()
)

## Save a diffusion list subjects

In [14]:
diffusion_subjects_df.to_csv("diffusion_subjects.csv", index=False)

## Save the long format df with all relevant data

In [15]:
merged.to_csv("merged.csv", index=False)

## Make a wide format for proper analysis of diffusion data

In [16]:
subject_cols = [
    "subjectID",
    "age_group",
    "strata",
    "Gender",
    "DDisc_AUC_40K",
    "Release",
    "Acquisition",
    "Age",
    "3T_Full_MR_Compl",
    "7T_Full_MR_Compl",
    "MEG_FullProt_Compl",
]

wide = (
    merged
    .pivot_table(
        index=subject_cols,
        columns="tractID",
        values=["dki_fa", "dki_md", "dki_mk", "dki_awf"],
        aggfunc="first",
    )
)

wide.columns = [f"{tract}_{metric}" for metric, tract in wide.columns]
wide = wide.reset_index()

In [18]:
wide.to_csv("../data/wide_diff_all_data.csv", index=False)

# Test combined df

In [19]:
all_subjects = pd.read_csv("../data/all_subjects_tract_means.csv")
test_split = pd.read_csv("../data/test_split.csv")

filtered_test = all_subjects[all_subjects["subjectID"].isin(test_split["Subject"])]

merged_test = filtered_test.merge(
    test_split,
    left_on="subjectID",
    right_on="Subject",
    how="left"
)

merged_test = merged_test.drop(columns=["Subject"])

## Counts

In [20]:
all_subjects["subjectID"].nunique()

1041

In [21]:
test_split["Subject"].nunique()

173

In [22]:
merged_test["subjectID"].nunique()

173

In [23]:
dd_df = pd.read_csv("../data/DD_AUC_targets.csv")


In [24]:
dd_df["Subject"].nunique()

889

## merge with DD data

In [25]:


merged_test = merged_test.merge(
    dd_df[["Subject", "DDisc_AUC_40K"]],
    left_on="subjectID",
    right_on="Subject",
    how="left"
)

merged_test = merged_test.drop(columns=["Subject"])

## Re-organize

In [26]:
cols = ["subjectID", "age_group", "strata", "Gender", "DDisc_AUC_40K"]
merged_test = merged_test[cols + [c for c in merged.columns if c not in cols]]

In [27]:
subject_cols = [
    "subjectID",
    "age_group",
    "strata",
    "Gender",
    "DDisc_AUC_40K",
    "Release",
    "Acquisition",
    "Age",
    "3T_Full_MR_Compl",
    "7T_Full_MR_Compl",
    "MEG_FullProt_Compl",
]

wide_test = (
    merged_test
    .pivot_table(
        index=subject_cols,
        columns="tractID",
        values=["dki_fa", "dki_md", "dki_mk", "dki_awf"],
        aggfunc="first",
    )
)

wide_test.columns = [f"{tract}_{metric}" for metric, tract in wide_test.columns]
wide_test = wide_test.reset_index()

In [28]:
wide_test.to_csv("../data/test_wide_diff_all_data.csv", index=False)